Some repos for reference:

https://github.com/lynguyenminh/CS114.M11.Vietnam-traffic-sign-detection

https://github.com/iamdinhthuan/Traffic-sign-VietNam-recognition

https://github.com/iamdinhthuan/Traffic-sign-VietNam-recognition/tree/main

In [1]:
# Install kagglehub
!pip install kagglehub

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set up Kaggle credentials using access token
import os
from getpass import getpass

# Input your Kaggle username and access token
kaggle_username = input("Enter your Kaggle username: ")
kaggle_token = getpass("Enter your Kaggle access token: ")

# Set environment variables for kagglehub
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_token

# Download dataset using kagglehub
import kagglehub
import shutil

# Download the dataset to a temporary location
print("Downloading dataset...")
dataset_path = kagglehub.dataset_download("maitam/vietnamese-traffic-signs")
print("Dataset downloaded to:", dataset_path)

# Copy to Google Drive so it persists
drive_dataset_path = "/content/drive/My Drive/vietnamese-traffic-signs"
if os.path.exists(drive_dataset_path):
    shutil.rmtree(drive_dataset_path)
shutil.copytree(dataset_path, drive_dataset_path)

print("✓ Dataset copied to Google Drive at:", drive_dataset_path)

Mounted at /content/drive
Using Colab cache for faster access to the 'vietnamese-traffic-signs' dataset.
Dataset downloaded to: /kaggle/input/vietnamese-traffic-signs
✓ Dataset copied to Google Drive at: /content/drive/My Drive/vietnamese-traffic-signs


In [2]:
# Use the dataset in your project
dataset_path = "/content/drive/My Drive/vietnamese-traffic-signs"

# List files in the dataset
import os
if os.path.exists(dataset_path):
    print("Files in dataset:")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # Show first 5 files per directory
            print(f'{subindent}{file}')
        if len(files) > 5:
            print(f'{subindent}... and {len(files) - 5} more files')
else:
    print(f"Dataset not found at {dataset_path}")

Files in dataset:
vietnamese-traffic-signs/
  archive/
    classes_en.txt
    classes.txt
    classes_vie.txt
    labels/
      1893.txt
      1711.txt
      0389.txt
      1773.txt
      0078.txt
      ... and 3211 more files
    images/
      0664.jpg
      1269.jpg
      2193.jpg
      0733.jpg
      2008.jpg
      ... and 3211 more files
    split_dataset/
      test_files.txt
      train_files.txt


In [3]:
# Helper function to parse YOLO format labels
import cv2
import matplotlib.pyplot as plt
import os

def parse_yolo_label(label_line, img_width, img_height):
    """
    Convert YOLO format to actual pixel coordinates
    Returns: class_id, x_center, y_center, width, height (in pixels)
    """
    parts = label_line.strip().split()
    class_id = int(parts[0])
    x_center_norm = float(parts[1])
    y_center_norm = float(parts[2])
    width_norm = float(parts[3])
    height_norm = float(parts[4])
    
    # Convert to pixel coordinates
    x_center = int(x_center_norm * img_width)
    y_center = int(y_center_norm * img_height)
    width = int(width_norm * img_width)
    height = int(height_norm * img_height)
    
    # Calculate bounding box corners
    x1 = int(x_center - width / 2)
    y1 = int(y_center - height / 2)
    x2 = int(x_center + width / 2)
    y2 = int(y_center + height / 2)
    
    return {
        'class_id': class_id,
        'x_center': x_center,
        'y_center': y_center,
        'width': width,
        'height': height,
        'x1': x1,
        'y1': y1,
        'x2': x2,
        'y2': y2
    }

def visualize_detections(image_path, label_path, class_names=None):
    """
    Display image with bounding boxes from YOLO label
    """
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Cannot read image: {image_path}")
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width = img_rgb.shape[:2]
    
    # Read labels
    if not os.path.exists(label_path):
        print(f"Label file not found: {label_path}")
        return
    
    with open(label_path, 'r') as f:
        labels = f.readlines()
    
    # Draw bounding boxes
    for label_line in labels:
        bbox = parse_yolo_label(label_line, width, height)
        x1, y1, x2, y2 = bbox['x1'], bbox['y1'], bbox['x2'], bbox['y2']
        class_id = bbox['class_id']
        
        # Draw box
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Add class label
        label_text = f"Class {class_id}"
        if class_names and class_id < len(class_names):
            label_text = class_names[class_id]
        cv2.putText(img_rgb, label_text, (x1, y1 - 10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Display
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title('Detected Objects')
    plt.tight_layout()
    plt.show()

# Example: Parse a label
example_label = "46 0.385417 0.537963 0.018750 0.035185"
bbox_info = parse_yolo_label(example_label, img_width=640, img_height=480)
print("Parsed label (với ảnh 640x480):")
print(f"  Class ID: {bbox_info['class_id']}")
print(f"  Tâm box: ({bbox_info['x_center']}, {bbox_info['y_center']})")
print(f"  Kích thước: {bbox_info['width']}x{bbox_info['height']}")
print(f"  Góc trái trên: ({bbox_info['x1']}, {bbox_info['y1']})")
print(f"  Góc phải dưới: ({bbox_info['x2']}, {bbox_info['y2']})")

Parsed label (với ảnh 640x480):
  Class ID: 46
  Tâm box: (246, 258)
  Kích thước: 12x16
  Góc trái trên: (240, 250)
  Góc phải dưới: (252, 266)


In [4]:
# Load class names from dataset
import yaml
import os

dataset_path = "/content/drive/My Drive/vietnamese-traffic-signs"

# Try to find and load class names
class_names = []

# Method 1: Look for data.yaml
yaml_path = os.path.join(dataset_path, "data.yaml")
if os.path.exists(yaml_path):
    with open(yaml_path, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
        if 'names' in data:
            class_names = data['names']
            print("✓ Loaded classes from data.yaml")

# Method 2: Look for classes.txt
if not class_names:
    classes_path = os.path.join(dataset_path, "classes.txt")
    if os.path.exists(classes_path):
        with open(classes_path, 'r', encoding='utf-8') as f:
            class_names = [line.strip() for line in f.readlines()]
            print("✓ Loaded classes from classes.txt")

# Method 3: Recursively search for class files
if not class_names:
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file in ['data.yaml', 'classes.txt', 'classes.names']:
                file_path = os.path.join(root, file)
                print(f"Found: {file_path}")
                if file == 'data.yaml':
                    with open(file_path, 'r', encoding='utf-8') as f:
                        data = yaml.safe_load(f)
                        if 'names' in data:
                            class_names = data['names']
                            break
                else:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        class_names = [line.strip() for line in f.readlines()]
                        break

# Display classes
if class_names:
    print(f"\n📊 Total classes: {len(class_names)}\n")
    for idx, name in enumerate(class_names):
        marker = "👉" if idx == 46 else "  "
        print(f"{marker} Class {idx}: {name}")
        if idx >= 50:  # Show first 51 classes
            print(f"   ... and {len(class_names) - 51} more classes")
            break
    
    # Show class 46 specifically
    if len(class_names) > 46:
        print(f"\n🎯 Class 46 = {class_names[46]}")
    else:
        print(f"\n⚠️  Class 46 not found in classes (total: {len(class_names)} classes)")
else:
    print("⚠️  No class names file found. Checking dataset structure...")

Found: /content/drive/My Drive/vietnamese-traffic-signs/archive/classes.txt

📊 Total classes: 52

   Class 0: W.224
   Class 1: W.205c
   Class 2: P.102
   Class 3: R.302a
   Class 4: W.205a
   Class 5: W.207
   Class 6: W.201a
   Class 7: P.123a
   Class 8: I.434a
   Class 9: R.303
   Class 10: P.130
   Class 11: I.409
   Class 12: R.415a
   Class 13: W.245a
   Class 14: P.106a*Xe tải
   Class 15: W.203c
   Class 16: P.117*
   Class 17: P.124a*
   Class 18: P.107
   Class 19: P.124d
   Class 20: P.103a
   Class 21: W.203b
   Class 22: W.221b
   Class 23: P.111
   Class 24: P.129
   Class 25: S.505a*Xe máy
   Class 26: W.246a
   Class 27: W.225
   Class 28: S.505a*Xe tải và công
   Class 29: P.104
   Class 30: S.505a*Xe tải
   Class 31: Camera
   Class 32: P.123b
   Class 33: W.202b
   Class 34: B.8a
   Class 35: P.137
   Class 36: P.139
   Class 37: W.205b
   Class 38: P.127*50
   Class 39: P.127*60
   Class 40: P.127*80
   Class 41: P.127*40
   Class 42: R.301e
   Class 43: W.239b*
 

In [5]:
# Giải thích mã biển báo giao thông Việt Nam

traffic_sign_meanings = {
    'P': {
        'name': 'Prohibition (Biển cấm)',
        'description': 'Những biển báo cấm hành động',
        'examples': ['P.131a - Cấm đỗ xe', 'P.102 - Cấm vào']
    },
    'W': {
        'name': 'Warning (Biển cảnh báo)',
        'description': 'Những biển báo cảnh báo nguy hiểm',
        'examples': ['W.224 - Cảnh báo đối tượng khác', 'W.201a - Cảnh báo giao lộ']
    },
    'R': {
        'name': 'Regulatory (Biển quy định)',
        'description': 'Những biển báo quy định giao thông',
        'examples': ['R.303 - Quy định tốc độ tối đa', 'R.302a - Quy định vận tốc']
    },
    'I': {
        'name': 'Information (Biển thông tin)',
        'description': 'Những biển báo thông tin hữu ích',
        'examples': ['I.409 - Thông tin bến xe', 'I.434a - Thông tin bãi đậu xe']
    }
}

print("=" * 70)
print("GIẢI THÍCH MÃ BIỂN BÁO GIAO THÔNG VIỆT NAM")
print("=" * 70)

for code, info in traffic_sign_meanings.items():
    print(f"\n🔹 {code} - {info['name']}")
    print(f"   Mô tả: {info['description']}")
    print(f"   Ví dụ: {', '.join(info['examples'])}")

print("\n" + "=" * 70)
print("🎯 CLASS 46 = P.131a")
print("=" * 70)
print("""
✓ P = Prohibition (Biển cấm)
✓ 131a = Loại cụ thể: CẤM ĐỖ XE

Ý nghĩa: Biển báo cấm đỗ/dừng xe ở khu vực này.
Hình dạng: Tròn, nền xanh/đỏ với dấu X hoặc ký hiệu cấm
Màu sắc: Đỏ/Xanh tùy theo quy định cụ thể

📌 Khi model detect class 46, nó đã nhận diện một biển cấm đỗ xe!
""")

# Summary: Load all classes and organize by type
print("\n" + "=" * 70)
print("DANH SÁCH ĐẦY ĐỦ CÁC BIỂN BÁO")
print("=" * 70)

if class_names:
    # Group by type
    by_type = {'P': [], 'W': [], 'R': [], 'I': [], 'Other': []}
    
    for idx, name in enumerate(class_names):
        prefix = name[0] if name else 'Other'
        if prefix in by_type:
            by_type[prefix].append((idx, name))
        else:
            by_type['Other'].append((idx, name))
    
    # Display organized
    for sign_type in ['P', 'W', 'R', 'I', 'Other']:
        if by_type[sign_type]:
            type_name = traffic_sign_meanings.get(sign_type, {}).get('name', sign_type)
            print(f"\n{sign_type} - {type_name}: ({len(by_type[sign_type])} loại)")
            for idx, name in by_type[sign_type][:10]:  # Show first 10
                marker = "👉" if idx == 46 else "  "
                print(f"  {marker} Class {idx}: {name}")
            if len(by_type[sign_type]) > 10:
                print(f"     ... và {len(by_type[sign_type]) - 10} loại khác")

GIẢI THÍCH MÃ BIỂN BÁO GIAO THÔNG VIỆT NAM

🔹 P - Prohibition (Biển cấm)
   Mô tả: Những biển báo cấm hành động
   Ví dụ: P.131a - Cấm đỗ xe, P.102 - Cấm vào

🔹 W - Warning (Biển cảnh báo)
   Mô tả: Những biển báo cảnh báo nguy hiểm
   Ví dụ: W.224 - Cảnh báo đối tượng khác, W.201a - Cảnh báo giao lộ

🔹 R - Regulatory (Biển quy định)
   Mô tả: Những biển báo quy định giao thông
   Ví dụ: R.303 - Quy định tốc độ tối đa, R.302a - Quy định vận tốc

🔹 I - Information (Biển thông tin)
   Mô tả: Những biển báo thông tin hữu ích
   Ví dụ: I.409 - Thông tin bến xe, I.434a - Thông tin bãi đậu xe

🎯 CLASS 46 = P.131a

✓ P = Prohibition (Biển cấm)
✓ 131a = Loại cụ thể: CẤM ĐỖ XE

Ý nghĩa: Biển báo cấm đỗ/dừng xe ở khu vực này.
Hình dạng: Tròn, nền xanh/đỏ với dấu X hoặc ký hiệu cấm
Màu sắc: Đỏ/Xanh tùy theo quy định cụ thể

📌 Khi model detect class 46, nó đã nhận diện một biển cấm đỗ xe!


DANH SÁCH ĐẦY ĐỦ CÁC BIỂN BÁO

P - Prohibition (Biển cấm): (22 loại)
     Class 2: P.102
     Class 7: P.123

In [6]:
# Load classes in both English and Vietnamese
import os

dataset_path = "/content/drive/My Drive/vietnamese-traffic-signs"

# Load English classes
classes_en = []
classes_en_path = os.path.join(dataset_path, "archive/classes_en.txt")
if os.path.exists(classes_en_path):
    with open(classes_en_path, 'r', encoding='utf-8') as f:
        classes_en = [line.strip() for line in f.readlines()]
    print(f"✓ Loaded {len(classes_en)} English class names")

# Load Vietnamese classes
classes_vie = []
classes_vie_path = os.path.join(dataset_path, "archive/classes_vie.txt")
if os.path.exists(classes_vie_path):
    with open(classes_vie_path, 'r', encoding='utf-8') as f:
        classes_vie = [line.strip() for line in f.readlines()]
    print(f"✓ Loaded {len(classes_vie)} Vietnamese class names")

# Display side by side
print("\n" + "=" * 100)
print(f"{'Class ID':<10} {'Code':<12} {'English':<40} {'Tiếng Việt':<40}")
print("=" * 100)

for idx in range(len(class_names)):
    code = class_names[idx]
    en_name = classes_en[idx] if idx < len(classes_en) else "N/A"
    vie_name = classes_vie[idx] if idx < len(classes_vie) else "N/A"
    
    marker = "👉" if idx == 46 else "  "
    print(f"{marker} {idx:<8} {code:<12} {en_name:<40} {vie_name:<40}")

print("\n" + "=" * 100)
print("🎯 CLASS 46 DETAILS")
print("=" * 100)
if 46 < len(class_names):
    print(f"Code:          {class_names[46]}")
    print(f"English:       {classes_en[46] if 46 < len(classes_en) else 'N/A'}")
    print(f"Tiếng Việt:    {classes_vie[46] if 46 < len(classes_vie) else 'N/A'}")
    print(f"\nMeaning: {classes_vie[46] if 46 < len(classes_vie) else 'N/A'}")
    print(f"In English: {classes_en[46] if 46 < len(classes_en) else 'N/A'}")

# Store for later use in visualization
class_names_en = classes_en
class_names_vie = classes_vie

print("\n✓ Ready to use bilingual class names in visualization functions!")

✓ Loaded 52 English class names
✓ Loaded 52 Vietnamese class names

Class ID   Code         English                                  Tiếng Việt                              
   0        W.224        Pedestrian Crossing                      Đường người đi bộ cắt ngang             
   1        W.205c       Equal-level Intersection                 Đường giao nhau (ngã ba bên phải)       
   2        P.102        No Entry                                 Cấm đi ngược chiều                      
   3        R.302a       Right Turn Only                          Phải đi vòng sang bên phải              
   4        W.205a       Intersection                             Giao nhau với đường đồng cấp            
   5        W.207        Intersection with a non-priority road    Giao nhau với đường không ưu tiên       
   6        W.201a       Danger zone on the left                  Chỗ ngoặt nguy hiểm vòng bên trái       
   7        P.123a       No Left Turn                             Cấm rẽ trái

In [9]:
# Enhanced visualization with bilingual labels
def visualize_detections_bilingual(image_path, label_path, language='vie', class_names_en=None, class_names_vie=None):
    """
    Display image with bounding boxes and bilingual labels
    language: 'vie' (tiếng Việt) or 'en' (English) or 'both'
    """
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Cannot read image: {image_path}")
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width = img_rgb.shape[:2]
    
    # Read labels
    if not os.path.exists(label_path):
        print(f"Label file not found: {label_path}")
        return
    
    with open(label_path, 'r') as f:
        labels = f.readlines()
    
    # Draw bounding boxes
    colors = {
        'P': (0, 0, 255),      # Red for Prohibition
        'W': (0, 165, 255),    # Orange for Warning
        'R': (0, 255, 255),    # Yellow for Regulatory
        'I': (0, 255, 0)       # Green for Information
    }
    
    for label_line in labels:
        bbox = parse_yolo_label(label_line, width, height)
        x1, y1, x2, y2 = bbox['x1'], bbox['y1'], bbox['x2'], bbox['y2']
        class_id = bbox['class_id']
        
        # Get color based on sign type
        sign_code = class_names[class_id] if class_id < len(class_names) else "?"
        sign_type = sign_code[0] if sign_code else '?'
        color = colors.get(sign_type, (255, 255, 255))
        
        # Draw box
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
        
        # Build label text
        if language == 'vie' and class_names_vie and class_id < len(class_names_vie):
            label_text = f"{sign_code}\n{class_names_vie[class_id]}"
        elif language == 'en' and class_names_en and class_id < len(class_names_en):
            label_text = f"{sign_code}\n{class_names_en[class_id]}"
        elif language == 'both':
            vie_name = class_names_vie[class_id] if class_names_vie and class_id < len(class_names_vie) else ""
            en_name = class_names_en[class_id] if class_names_en and class_id < len(class_names_en) else ""
            label_text = f"{sign_code}\n{vie_name}\n{en_name}"
        else:
            label_text = f"Class {class_id}"
        
        # Add label with background
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.5
        thickness = 1
        
        # For multi-line text
        for i, line in enumerate(label_text.split('\n')):
            y_offset = y1 - 10 - (i * 20)
            cv2.putText(img_rgb, line, (x1, y_offset), font, font_scale, color, thickness)
    
    # Display
    plt.figure(figsize=(14, 10))
    plt.imshow(img_rgb)
    plt.axis('off')
    lang_text = "Vietnamese" if language == 'vie' else "English" if language == 'en' else "Bilingual"
    plt.title(f'Traffic Sign Detection ({lang_text})')
    plt.tight_layout()
    plt.show()

# Example: Create a sample label file and visualize
print("Functions ready! Use like this:")
print("  visualize_detections_bilingual(image_path, label_path, language='vie')")
print("  visualize_detections_bilingual(image_path, label_path, language='en')")
print("  visualize_detections_bilingual(image_path, label_path, language='both')")

Functions ready! Use like this:
  visualize_detections_bilingual(image_path, label_path, language='vie')
  visualize_detections_bilingual(image_path, label_path, language='en')
  visualize_detections_bilingual(image_path, label_path, language='both')
